# Example 04: Automated calculation of melting temperature

In Example 03, we calculated the melting temperature for Cu manually by running a solid and a liquid reversible-scaling sweep and locating the free-energy crossing. Calphy can do the same thing in a fully automated way using `mode: melting_temperature`.

The EAM potential we will use is [Mishin, Y., M. J. Mehl, D. A. Papaconstantopoulos, A. F. Voter, and J. D. Kress. "Structural Stability and Lattice Defects in Copper: Ab Initio, Tight-Binding, and Embedded-Atom Calculations." Physical Review B 63, no. 22 (May 21, 2001): 224106.](https://doi.org/10.1103/PhysRevB.63.224106)

The calculation block gives the input conditions at which the calculation is carried out. First of all, the `mode` is `melting_temperature`. This mode is special and needs much less input than the other common modes — options such as `lattice` and `temperature` are (generally; see below for specific cases) not needed. The input file used in this example is:

```yaml
calculations:
- element: Cu
  mass: 63.546
  repeat:
  - 10
  - 10
  - 10
  md:
    timestep: 0.001
  mode: melting_temperature
  n_equilibration_steps: 10000
  n_iterations: 1
  n_print_steps: 100
  n_switching_steps: 25000
  pair_coeff: '* * ../potentials/Cu01.eam.alloy Cu'
  pair_style: eam/alloy
  equilibration_control: berendsen
  queue:
    commands:
    - conda activate calphy
    cores: 4
    scheduler: local
  melting_temperature:
    step: 400
```

Once the input file is set up, the calculation is run as usual:

```
calphy -i input.yaml
```

A log file (named after the calculation identifier, e.g. `<identifier>.log`) is produced with detailed information about the run. The most important lines are prefixed with `STATE`. Running `grep STATE *.log` gives, for example:

```
STATE: Temperature range of 957.770000-1757.770000 K
STATE: Tm = 1285.24 K +/- 0.24 K
```

The calculated melting temperature for this interatomic potential is about 1285 K (the experimental value for Cu is roughly 1357 K).

### How does it work?

Starting from an initial guess $T_g$ — the experimental melting point of the first element (looked up via [mendeleev](https://mendeleev.readthedocs.io/)), or `melting_temperature.guess` if you set it — calphy runs a solid and a liquid reversible-scaling sweep over the window $[T_g - \Delta T,\; T_g + \Delta T]$, whose half-width is `melting_temperature.step`:

- If the solid melts (or the liquid freezes) during its sweep, the window is shifted down (or up) and the pair of sweeps is rerun.
- Once both phases stay stable across the window, $T_m$ is the temperature at which the solid and liquid free-energy curves cross.
- If the crossing lies at the edge of the window, calphy linearly extrapolates the two free-energy curves to predict $T_m$, recentres the window on that prediction, and tries again.

The search stops when the crossing is found inside the window, or after `melting_temperature.attempts` attempts (default 5).

### FAQs

- **How can I tune the initial guess temperature?**
  Set `melting_temperature.guess` (in Kelvin) in the calculation block. If omitted, calphy uses the experimental melting point of the first element from [mendeleev](https://mendeleev.readthedocs.io/).

- **How can I tune the width of the temperature range?**
  Set `melting_temperature.step` — the half-width of the search window (default 200 K). For systems with significant MD hysteresis (small cells, slow-nucleating potentials), a larger value such as 400 K gives both phases room to sample cleanly on each side of $T_m$.

- **What if the system undergoes a solid-solid phase transition before melting?**
  The lattice that calphy automatically chooses for the solid is the **ground state**. For some elements — for example Ti, where HCP transforms to BCC and then to liquid — a solid-solid transition occurs before melting. To use the automated method in that case, specify the (higher-temperature) solid lattice explicitly with the `lattice` keyword.

- **How can I calculate the melting temperature at non-zero pressure?**
  Specify the required pressure with the `pressure` keyword. Setting `melting_temperature.guess` closer to the expected $T_m$ can help speed up the calculation, since the mendeleev value is for ambient pressure.
